# CV Assignment 2 - Plant disease: SAM2 ROI + ResNet18 + Grad-CAM (Colab)
**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Upload `CV_A2_PlantDisease.zip` to the Colab file panel (left, folder icon).
Run the cells top to bottom. Each step writes into `outputs/`; the last cells build the report and download everything.

In [1]:
!unzip -q -o /content/CV_A2_PlantDisease.zip -d /content
%cd /content/cv_a2
!ls

unzip:  cannot find or open /content/CV_A2_PlantDisease.zip, /content/CV_A2_PlantDisease.zip.zip or /content/CV_A2_PlantDisease.zip.ZIP.
[Errno 2] No such file or directory: '/content/cv_a2'
/Users/almukhammedamantay/Downloads/cv_a2/notebooks
A2_Colab.ipynb


## 1. Install

In [ ]:
!pip -q install gradio python-docx huggingface_hub opencv-python-headless tqdm scikit-learn
# SAM2 (needs torch >= 2.5; if pip complains run: !pip install -U torch torchvision  and restart the runtime)
!SAM2_BUILD_CUDA=0 pip -q install git+https://github.com/facebookresearch/sam2.git
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

## 2. SAM2 leaf masks for the whole dataset (about 1-3 min on T4)

In [ ]:
!python src/segment_sam2.py --data dataset --out outputs/seg --method sam2
from IPython.display import Image, display
display(Image('outputs/seg/figures/mask_examples.png', width=520))

## 3. Mask quality (IoU) - needs ~14 hand-drawn masks
1. Run the next cell and download `gt_todo.zip`.
2. In any editor (Paint, Photopea...) paint the **leaf white and the background black**, save as PNG with the **same file name** (`12.jpeg` -> `12.png`) into `gt_masks/diseased/` or `gt_masks/healthy/`.
3. Zip the `gt_masks` folder as `gt_masks.zip`, upload it to Colab and run the last cell of this section.

In [ ]:
!python src/eval_iou.py --prepare 14
!zip -qr gt_todo.zip gt_todo
from google.colab import files; files.download('gt_todo.zip')

In [ ]:
# after uploading gt_masks.zip:
!unzip -q -o /content/gt_masks.zip -d /content/cv_a2
!python src/eval_iou.py

## 4. Experiments: raw (A1) vs SAM2 masked vs SAM2 crop, 3 seeds each (about 5-10 min)

In [ ]:
!python src/run_experiments.py --data dataset --masks outputs/seg/masks --a1-metrics a1/metrics.json
import pandas as pd; display(pd.read_csv('outputs/results_table.csv'))
display(Image('outputs/figures/ablation_f1.png', width=480))

## 5. Explainability figure + latency

In [ ]:
!python src/make_figures.py
!python src/benchmark_latency.py
display(Image('outputs/figures/gradcam_examples.png', width=700))

## 6. Launch the demo app
The cell prints a public `https://xxxx.gradio.live` link (valid ~72 h while this cell runs). Take a screenshot for the report
and save it as `report/assets/app_screenshot.png` (upload it into that folder via the file panel). **Stop the cell afterwards.**

In [ ]:
!python src/app.py --share

## 7. Build the report and download everything
Edit the three links (repo / video / app), then run. Open the .docx in Word, check the red `[...]` placeholders, export to PDF.

In [ ]:
!python report/build_report.py --repo-url "https://github.com/YOUR_NAME/YOUR_REPO" --video-url "VIDEO_LINK" --app-url "APP_LINK"
!zip -qr results.zip outputs Report_A2_Plant_Disease_SAM2.docx
from google.colab import files
files.download('Report_A2_Plant_Disease_SAM2.docx'); files.download('results.zip')